In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType

spark = SparkSession.builder.appName("Jupyter").getOrCreate()

conf = spark.sparkContext.getConf()

# Filter and print configurations that start with 'spark.sql'
print("Spark SQL Configurations:")
for key, value in conf.getAll():
    if key.startswith("spark.sql"):
        print(f"{key} = {value}")

print(f"spark.sql.catalog.data.s3.endpoint: {spark.conf.get('spark.sql.catalog.data.s3.endpoint', 'Not Set')}")

for key, value in conf.getAll():
    if key.startswith("spark.hadoop"):
        print(f"{key} = {value}")

for key, value in conf.getAll():
    print(f"{key} = {value}")
spark

In [ ]:
namespace1 = "db"
table1 = f"{namespace1}.test"
namespace2 = "cataglog"
table2 = f"{namespace1}.catalog_test"


In [ ]:
spark.sql("CREATE NAMESPACE if Not exists db")
spark.sql("CREATE NAMESPACE if Not exists catalog")


In [ ]:
spark.sql("CREATE NAMESPACE if Not exists silver_data.db")

In [ ]:
data = [("James","","Smith","36636","M",3000),
    ("Michael","Rose","","40288","M",4000),
    ("Robert","","Williams","42114","M",4000),
    ("Maria","Anne","Jones","39192","F",4000),
    ("Jen","Mary","Brown","","F",-1)
  ]

schema = StructType([ \
    StructField("firstname",StringType(),True), \
    StructField("middlename",StringType(),True), \
    StructField("lastname",StringType(),True), \
    StructField("id", StringType(), True), \
    StructField("gender", StringType(), True), \
    StructField("salary", IntegerType(), True) \
  ])

data

df = spark.createDataFrame(data=data, schema=schema)
df.printSchema()


In [4]:
df.writeTo(table1).createOrReplace()

In [ ]:
df.writeTo(table2).createOrReplace()

In [ ]:
spark.sql("CREATE TABLE data.db.my_new_table ( \
  id INT, \
  name STRING\
) USING iceberg")

In [ ]:
res = spark.sql(f"SELECT * FROM {table1}")
print(f"schema: {res.schema}")
res.show()

In [ ]:
res = spark.sql(f"SELECT * FROM {table2}")
res.show()

In [ ]:
df.writeTo("silver_data.db.test2").createOrReplace()

In [ ]:
#Inspecting the history of the table:
res = spark.sql(f"SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor FROM {table1}.history")
res.show(truncate=False)

In [ ]:
#inspect the table snapshots
res = spark.sql(f"SELECT committed_at, snapshot_id, operation, manifest_list, summary FROM {table1}.snapshots")
res.show(truncate=False)


In [ ]:
#query to get back the data files
res = spark.sql(f"SELECT file_path, file_format, record_count FROM db.pg_catalog.files")
res.show(truncate=False)

In [ ]:
#query the manifests
res = spark.sql(f"SELECT length, path, added_data_files_count, added_snapshot_id FROM {table1}.manifests")
res.show(truncate=False)

In [ ]:
#query the metadata log entries
res = spark.sql(f"SELECT timestamp, file, latest_snapshot_id, latest_schema_id, latest_sequence_number FROM {table1}.metadata_log_entries")
res.show(truncate=False)

In [ ]:
spark.stop()